# Transformer Architectures -- Advanced

> **Section 01** -- Topic 04 -- `advanced` 
> **Prerequisites:** `foundations.ipynb` 
> **Objective:** Implement the modern components that replaced every classical piece of the original transformer: RoPE, ALiBi, RMSNorm, SwiGLU, parallel layers, and assemble the Llama 3 recipe.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np
import time

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 1. Rotary Position Embeddings (RoPE)

RoPE (Su et al., 2021) encodes position information by **rotating** query and key vectors in 2D subspaces. Instead of adding a position embedding, we multiply Q and K by rotation matrices.

### The core idea

For a pair of dimensions $(x_i, x_{i+1})$ at position $m$, RoPE applies:

$$\begin{pmatrix} x_i' \\ x_{i+1}' \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} x_i \\ x_{i+1} \end{pmatrix}$$

where $\theta_i = 10000^{-2i/d}$.

### Why RoPE is elegant

The dot product between two rotated vectors depends only on their **relative** position:

$$\langle \text{RoPE}(q, m), \text{RoPE}(k, n) \rangle = f(q, k, m - n)$$

This gives us relative position encoding without any learned parameters, and it generalizes to longer sequences than seen during training.

In [ ]:
def precompute_rope_frequencies(d_model: int, max_len: int = 4096, theta: float = 10000.0):
    """Precompute the complex exponentials for RoPE.
    
    Returns cos and sin tensors of shape (max_len, d_model//2).
    """
    # Frequency for each dimension pair
    freqs = 1.0 / (theta ** (torch.arange(0, d_model, 2).float() / d_model))
    # Position indices
    positions = torch.arange(max_len).float()
    # Outer product: (max_len, d_model//2)
    angles = torch.outer(positions, freqs)
    return torch.cos(angles), torch.sin(angles)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply RoPE to input tensor x of shape (B, n_heads, T, d_k)."""
    B, H, T, d_k = x.shape
    # Reshape x into pairs: (B, H, T, d_k//2, 2)
    x_pairs = x.float().reshape(B, H, T, d_k // 2, 2)
    x_real = x_pairs[..., 0]  # (B, H, T, d_k//2)
    x_imag = x_pairs[..., 1]
    
    # Slice cos/sin to match sequence length
    cos_t = cos[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, d_k//2)
    sin_t = sin[:T].unsqueeze(0).unsqueeze(0)
    
    # Apply rotation
    out_real = x_real * cos_t - x_imag * sin_t
    out_imag = x_real * sin_t + x_imag * cos_t
    
    # Interleave back
    out = torch.stack([out_real, out_imag], dim=-1).reshape(B, H, T, d_k)
    return out.type_as(x)


# Test RoPE
d_k = 64
cos_cached, sin_cached = precompute_rope_frequencies(d_k, max_len=1024)

q = torch.randn(2, 8, 32, d_k)  # (batch, heads, seq_len, d_k)
k = torch.randn(2, 8, 32, d_k)

q_rot = apply_rope(q, cos_cached, sin_cached)
k_rot = apply_rope(k, cos_cached, sin_cached)

print(f"Input Q shape: {q.shape}")
print(f"Rotated Q shape: {q_rot.shape}")
print(f"Norms preserved: |q|={q[0,0,0].norm():.4f}, |q_rot|={q_rot[0,0,0].norm():.4f}")

In [ ]:
# Verify relative position property: dot product depends on (m - n)
d_k = 64
cos_c, sin_c = precompute_rope_frequencies(d_k, max_len=256)

# Create identical q and k vectors at different positions
q_vec = torch.randn(1, 1, 1, d_k).expand(1, 1, 100, d_k).clone()  # same vector, 100 copies
k_vec = torch.randn(1, 1, 1, d_k).expand(1, 1, 100, d_k).clone()

# Apply RoPE - each copy gets different rotation based on position
q_rotated = apply_rope(q_vec, cos_c, sin_c)
k_rotated = apply_rope(k_vec, cos_c, sin_c)

# Compute dot products for different relative distances
pos_m = 50  # query position
distances = list(range(-20, 21))
dot_products = []
for d in distances:
    pos_n = pos_m + d
    if 0 <= pos_n < 100:
        dot = (q_rotated[0, 0, pos_m] * k_rotated[0, 0, pos_n]).sum().item()
        dot_products.append(dot)
    else:
        dot_products.append(float('nan'))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(distances, dot_products, 'o-', linewidth=2, markersize=4)
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='Same position')
ax.set_xlabel('Relative Position (m - n)', fontsize=12)
ax.set_ylabel('Dot Product q(m) . k(n)', fontsize=12)
ax.set_title('RoPE: Attention Score Depends on Relative Position', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize RoPE rotation frequencies
d_model = 128
cos_vis, sin_vis = precompute_rope_frequencies(d_model, max_len=512)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of cosine values
im = axes[0].imshow(cos_vis[:128, :32].numpy(), aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xlabel('Dimension pair index')
axes[0].set_ylabel('Position')
axes[0].set_title('RoPE Cosine Components')
plt.colorbar(im, ax=axes[0])

# Frequency spectrum
freqs = 1.0 / (10000.0 ** (torch.arange(0, d_model, 2).float() / d_model))
wavelengths = 2 * math.pi / freqs
axes[1].semilogy(range(len(freqs)), wavelengths.numpy(), 'o-', markersize=3)
axes[1].set_xlabel('Dimension pair index')
axes[1].set_ylabel('Wavelength (positions)')
axes[1].set_title('RoPE Wavelengths per Dimension')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=512, color='red', linestyle='--', alpha=0.5, label='Context length = 512')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Shortest wavelength: {wavelengths.min():.1f} positions")
print(f"Longest wavelength: {wavelengths.max():.0f} positions")

---
## 2. ALiBi (Attention with Linear Biases)

ALiBi (Press et al., 2022) takes a radically different approach: no position embeddings at all. Instead, add a **linear bias** to attention scores based on the distance between query and key positions.

$$\text{score}(q_i, k_j) = q_i \cdot k_j - m \cdot |i - j|$$

where $m$ is a head-specific slope. Slopes are set geometrically: $m = 2^{-8/H}, 2^{-16/H}, ..., 2^{-8}$ for $H$ heads.

### ALiBi vs RoPE

| Property | RoPE | ALiBi |
|----------|------|-------|
| Mechanism | Rotate Q,K | Bias attention scores |
| Parameters | None | None |
| Length generalization | Good with NTK scaling | Excellent (designed for it) |
| Adopted by | Llama, Mistral, most modern LLMs | BLOOM, MPT |

In [ ]:
def get_alibi_slopes(n_heads: int) -> torch.Tensor:
    """Compute ALiBi slopes for each attention head.
    
    Returns slopes of shape (n_heads,).
    """
    # For power-of-2 heads, use geometric sequence
    # For non-power-of-2, interpolate
    def _get_slopes_power_of_2(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]
    
    if math.log2(n_heads).is_integer():
        slopes = _get_slopes_power_of_2(n_heads)
    else:
        # Get the nearest power of 2 above and below
        closest_power = 2 ** math.floor(math.log2(n_heads))
        slopes = _get_slopes_power_of_2(closest_power)
        extra = _get_slopes_power_of_2(2 * closest_power)
        slopes = slopes + extra[0::2][:n_heads - closest_power]
    
    return torch.tensor(slopes, dtype=torch.float32)


def build_alibi_bias(n_heads: int, seq_len: int) -> torch.Tensor:
    """Build the ALiBi attention bias matrix.
    
    Returns: (n_heads, seq_len, seq_len)
    """
    slopes = get_alibi_slopes(n_heads)  # (n_heads,)
    # Distance matrix: |i - j|
    positions = torch.arange(seq_len)
    distance = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs().float()  # (T, T)
    # Apply slopes: (n_heads, T, T)
    alibi = -slopes.unsqueeze(1).unsqueeze(2) * distance.unsqueeze(0)
    return alibi


# Visualize ALiBi biases
n_heads = 8
seq_len = 64
alibi = build_alibi_bias(n_heads, seq_len)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
slopes = get_alibi_slopes(n_heads)
for i, ax in enumerate(axes.flat):
    im = ax.imshow(alibi[i].numpy(), cmap='RdBu_r', aspect='auto')
    ax.set_title(f'Head {i} (slope={slopes[i]:.4f})', fontsize=10)
    ax.set_xlabel('Key pos')
    ax.set_ylabel('Query pos')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('ALiBi Attention Biases per Head', fontsize=14)
plt.tight_layout()
plt.show()

print(f"ALiBi slopes: {slopes.tolist()}")
print(f"Steepest head penalizes distance-10 by: {-slopes.max().item() * 10:.2f}")
print(f"Gentlest head penalizes distance-10 by: {-slopes.min().item() * 10:.4f}")

In [ ]:
class ALiBiAttention(nn.Module):
    """Multi-head attention with ALiBi position encoding."""
    
    def __init__(self, d_model: int, n_heads: int, max_len: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_qkv = nn.Linear(d_model, 3 * d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Precompute ALiBi biases
        alibi = build_alibi_bias(n_heads, max_len)
        self.register_buffer('alibi', alibi)  # (n_heads, max_len, max_len)
    
    def forward(self, x, causal_mask=None):
        B, T, C = x.shape
        qkv = self.W_qkv(x).reshape(B, T, 3, self.n_heads, self.d_k).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]  # each: (B, n_heads, T, d_k)
        
        # Standard attention + ALiBi bias (no position embeddings needed!)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores + self.alibi[:, :T, :T].unsqueeze(0)  # add ALiBi
        
        if causal_mask is not None:
            scores = scores.masked_fill(causal_mask[:, :, :T, :T] == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(context)


# Compare attention patterns
alibi_attn = ALiBiAttention(d_model=128, n_heads=8)
x = torch.randn(1, 32, 128)
T = 32
causal_mask = torch.tril(torch.ones(T, T)).unsqueeze(0).unsqueeze(0)
out = alibi_attn(x, causal_mask)
print(f"ALiBi attention output shape: {out.shape}")
print(f"No position embeddings needed -- positions are encoded via attention biases!")

---
## 3. RMSNorm

RMSNorm (Zhang & Sennrich, 2019) simplifies LayerNorm by removing the mean-centering step. It normalizes by the **root mean square** of activations only.

$$\text{RMSNorm}(x) = \gamma \cdot \frac{x}{\text{RMS}(x) + \epsilon}, \quad \text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2}$$

### Why models switched from LayerNorm

1. **Faster**: No mean computation, no mean subtraction (saves 2 operations per element)
2. **No learned bias**: Only $\gamma$ (scale), no $\beta$ (shift) -- fewer parameters
3. **Empirically equivalent**: No degradation in model quality
4. **Better for hardware**: Simpler reduction operation

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))  # gamma only, no beta
        self.eps = eps
    
    def forward(self, x):
        # RMS = sqrt(mean(x^2))
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)


# Compare RMSNorm vs LayerNorm outputs
d_model = 256
x = torch.randn(4, 32, d_model)

rms_norm = RMSNorm(d_model)
layer_norm = nn.LayerNorm(d_model)

rms_out = rms_norm(x)
ln_out = layer_norm(x)

print(f"Input stats:    mean={x.mean():.4f}, std={x.std():.4f}")
print(f"RMSNorm stats:  mean={rms_out.mean():.4f}, std={rms_out.std():.4f}")
print(f"LayerNorm stats: mean={ln_out.mean():.4f}, std={ln_out.std():.4f}")
print(f"\nRMSNorm params: {sum(p.numel() for p in rms_norm.parameters())} (only gamma)")
print(f"LayerNorm params: {sum(p.numel() for p in layer_norm.parameters())} (gamma + beta)")

In [ ]:
# Benchmark RMSNorm vs LayerNorm
d_model = 1024
x = torch.randn(16, 256, d_model)

rms = RMSNorm(d_model)
ln = nn.LayerNorm(d_model)

def benchmark(fn, x, name, n_iters=500):
    # Warmup
    for _ in range(50):
        _ = fn(x)
    start = time.perf_counter()
    for _ in range(n_iters):
        _ = fn(x)
    elapsed = time.perf_counter() - start
    ms_per_iter = elapsed / n_iters * 1000
    print(f"{name}: {ms_per_iter:.3f} ms/iter")
    return ms_per_iter

t_rms = benchmark(rms, x, "RMSNorm")
t_ln = benchmark(ln, x, "LayerNorm")

print(f"\nRMSNorm is {t_ln/t_rms:.2f}x {'faster' if t_rms < t_ln else 'slower'} than LayerNorm")

In [ ]:
# Visualize what each normalization does to the distribution
x = torch.randn(1000, 256) * 3 + 2  # shifted, scaled distribution

rms_norm_vis = RMSNorm(256)
ln_vis = nn.LayerNorm(256)

with torch.no_grad():
    rms_out = rms_norm_vis(x)
    ln_out = ln_vis(x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(x.flatten().numpy(), bins=100, alpha=0.7, density=True, color='gray')
axes[0].set_title(f'Input (mean={x.mean():.2f}, std={x.std():.2f})')
axes[0].set_xlim(-10, 15)

axes[1].hist(ln_out.flatten().numpy(), bins=100, alpha=0.7, density=True, color='#2196F3')
axes[1].set_title(f'LayerNorm (mean={ln_out.mean():.2f}, std={ln_out.std():.2f})')
axes[1].set_xlim(-5, 5)

axes[2].hist(rms_out.flatten().numpy(), bins=100, alpha=0.7, density=True, color='#FF9800')
axes[2].set_title(f'RMSNorm (mean={rms_out.mean():.2f}, std={rms_out.std():.2f})')
axes[2].set_xlim(-5, 5)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle('LayerNorm centers to zero mean; RMSNorm only scales magnitude', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. SwiGLU Activation

SwiGLU (Shazeer, 2020) replaces the standard FFN with a **gated linear unit** using the SiLU (Swish) activation:

$$\text{SwiGLU}(x) = (\text{SiLU}(xW_1) \odot xW_3) W_2$$

where $\odot$ is element-wise multiplication and $\text{SiLU}(x) = x \cdot \sigma(x)$.

### Why three weight matrices?

Standard FFN: $W_1$ (up-project) and $W_2$ (down-project) -- 2 matrices.

SwiGLU: $W_1$ (gate), $W_3$ (value), $W_2$ (down-project) -- 3 matrices.

The gate ($W_1$) learns **which features to let through**, while $W_3$ provides the actual values. To keep parameter count similar, the hidden dimension is reduced from $4d$ to $\frac{8}{3}d$ (Llama uses this ratio).

In [ ]:
class SwiGLU(nn.Module):
    """SwiGLU feed-forward network as used in Llama, Mistral, etc."""
    
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.0):
        super().__init__()
        # Default: 8/3 * d_model, rounded to nearest multiple of 256
        if d_ff is None:
            d_ff = int(8 / 3 * d_model)
            d_ff = ((d_ff + 255) // 256) * 256  # round up
        
        self.w1 = nn.Linear(d_model, d_ff, bias=False)  # gate
        self.w3 = nn.Linear(d_model, d_ff, bias=False)  # value
        self.w2 = nn.Linear(d_ff, d_model, bias=False)  # down projection
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        gate = F.silu(self.w1(x))  # SiLU = x * sigmoid(x)
        value = self.w3(x)
        return self.dropout(self.w2(gate * value))


class StandardFFN(nn.Module):
    """Standard FFN for comparison."""
    def __init__(self, d_model: int, d_ff: int = None):
        super().__init__()
        d_ff = d_ff or d_model * 4
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.w2(F.gelu(self.w1(x)))


d_model = 256
swiglu = SwiGLU(d_model)
standard = StandardFFN(d_model)

swiglu_params = sum(p.numel() for p in swiglu.parameters())
standard_params = sum(p.numel() for p in standard.parameters())

print(f"Standard FFN (4x):  d_ff={d_model*4}, params={standard_params:,}")
print(f"SwiGLU (8/3x):     d_ff={swiglu.w1.out_features}, params={swiglu_params:,}")
print(f"Parameter ratio: {swiglu_params/standard_params:.3f}")
print(f"\nSwiGLU uses 3 matrices but smaller hidden dim -> roughly same param count")

In [ ]:
# Visualize SiLU (Swish) and the gating mechanism
x = torch.linspace(-5, 5, 300)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# SiLU vs GELU vs ReLU
axes[0].plot(x.numpy(), F.silu(x).numpy(), label='SiLU (Swish)', linewidth=2)
axes[0].plot(x.numpy(), F.gelu(x).numpy(), label='GELU', linewidth=2)
axes[0].plot(x.numpy(), F.relu(x).numpy(), label='ReLU', linewidth=2, linestyle='--')
axes[0].set_title('Activation Functions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axvline(x=0, color='k', linewidth=0.5)

# Gating visualization: show how gate modulates values
gate_input = torch.randn(100)
gate_output = F.silu(gate_input)
value_input = torch.randn(100)
gated_output = gate_output * value_input

axes[1].scatter(value_input.numpy(), gated_output.numpy(), alpha=0.5, s=10)
axes[1].set_xlabel('Value (W3 output)')
axes[1].set_ylabel('Gated output (SiLU(W1) * W3)')
axes[1].set_title('SwiGLU Gating Effect')
axes[1].grid(True, alpha=0.3)

# Gate activation distribution
gate_vals = F.silu(torch.randn(10000))
axes[2].hist(gate_vals.numpy(), bins=100, alpha=0.7, density=True)
axes[2].set_title('Distribution of SiLU Gate Values')
axes[2].set_xlabel('Gate value')
axes[2].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Parallel Attention + FFN

PaLM (Chowdhery et al., 2022) introduced **parallel** attention and FFN computation:

```
Standard (serial):    x = x + Attn(Norm(x));  x = x + FFN(Norm(x))
Parallel (PaLM):      x = x + Attn(Norm(x)) + FFN(Norm(x))
```

### Why parallel?

1. **Throughput**: On TPUs/GPUs, attention and FFN can execute simultaneously
2. **One norm instead of two**: The same normalized input feeds both sublayers
3. **Minimal quality impact**: At large scale, the quality difference is negligible
4. **~15% speedup** reported in PaLM paper

In [ ]:
class MultiHeadAttentionSimple(nn.Module):
    """Simplified MHA for use in block comparisons."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_model = d_model
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.W_qkv(x).reshape(B, T, 3, self.n_heads, self.d_k).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(out)


class SerialBlock(nn.Module):
    """Standard serial block: Attn then FFN."""
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = MultiHeadAttentionSimple(d_model, n_heads)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)
    
    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class ParallelBlock(nn.Module):
    """PaLM-style parallel block: Attn and FFN share the same normed input."""
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm = RMSNorm(d_model)  # single norm!
        self.attn = MultiHeadAttentionSimple(d_model, n_heads)
        self.ffn = SwiGLU(d_model, d_ff)
    
    def forward(self, x, mask=None):
        normed = self.norm(x)
        # Both sublayers operate on the same normed input
        x = x + self.attn(normed, mask) + self.ffn(normed)
        return x


d_model, n_heads, d_ff = 256, 8, 512
serial = SerialBlock(d_model, n_heads, d_ff)
parallel = ParallelBlock(d_model, n_heads, d_ff)

print(f"Serial block params:   {sum(p.numel() for p in serial.parameters()):,}")
print(f"Parallel block params: {sum(p.numel() for p in parallel.parameters()):,}")
print(f"Parallel saves: {sum(p.numel() for p in serial.parameters()) - sum(p.numel() for p in parallel.parameters()):,} params (one fewer LayerNorm)")

In [ ]:
# Benchmark serial vs parallel
x = torch.randn(8, 128, d_model)
T = 128
mask = torch.tril(torch.ones(T, T)).unsqueeze(0).unsqueeze(0)

t_serial = benchmark(lambda inp: serial(inp, mask), x, "Serial block")
t_parallel = benchmark(lambda inp: parallel(inp, mask), x, "Parallel block")

print(f"\nParallel is {t_serial/t_parallel:.2f}x {'faster' if t_parallel < t_serial else 'slower'}")
print("Note: Parallel gains are more pronounced on GPUs with true parallel execution")

---
## 6. The Modern Recipe: Llama 3 Architecture

Llama 3 (Meta, 2024) combines all the modern components we have built:

| Component | Classical | Llama 3 |
|-----------|-----------|----------|
| Position encoding | Sinusoidal / Learned | **RoPE** |
| Normalization | Post-norm LayerNorm | **Pre-norm RMSNorm** |
| FFN activation | ReLU / GELU | **SwiGLU** |
| Attention | Multi-Head | **Grouped-Query Attention (GQA)** |
| FFN expansion | 4x | **~2.7x** (adjusted for 3 matrices) |
| Bias | Yes | **No** (no bias anywhere) |

Let us assemble the complete architecture.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """Grouped-Query Attention (GQA) as used in Llama 2/3.
    
    n_kv_heads < n_heads: multiple query heads share the same K,V head.
    n_kv_heads = 1: Multi-Query Attention (MQA)
    n_kv_heads = n_heads: standard Multi-Head Attention (MHA)
    """
    
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, max_len: int = 4096):
        super().__init__()
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads  # how many Q heads share each KV head
        self.d_k = d_model // n_heads
        self.d_model = d_model
        
        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(n_heads * self.d_k, d_model, bias=False)
        
        # RoPE
        cos, sin = precompute_rope_frequencies(self.d_k, max_len)
        self.register_buffer('cos_cached', cos)
        self.register_buffer('sin_cached', sin)
    
    def _repeat_kv(self, x):
        """Repeat KV heads to match number of query heads."""
        if self.n_rep == 1:
            return x
        B, H, T, D = x.shape
        return x[:, :, None, :, :].expand(B, H, self.n_rep, T, D).reshape(B, H * self.n_rep, T, D)
    
    def forward(self, x, mask=None):
        B, T, C = x.shape
        
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        
        # Apply RoPE to Q and K
        Q = apply_rope(Q, self.cos_cached, self.sin_cached)
        K = apply_rope(K, self.cos_cached, self.sin_cached)
        
        # Expand KV heads to match Q heads
        K = self._repeat_kv(K)
        V = self._repeat_kv(V)
        
        # Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(out)


# Show KV cache savings
d_model = 256
n_heads = 8

configs = [
    ("MHA (n_kv=8)", 8),
    ("GQA (n_kv=4)", 4),
    ("GQA (n_kv=2)", 2),
    ("MQA (n_kv=1)", 1),
]

print(f"{'Config':<20} {'Q params':>10} {'K params':>10} {'V params':>10} {'Total KV':>10} {'KV savings':>12}")
print("-" * 75)
baseline_kv = None
for name, n_kv in configs:
    gqa = GroupedQueryAttention(d_model, n_heads, n_kv)
    q_p = gqa.W_q.weight.numel()
    k_p = gqa.W_k.weight.numel()
    v_p = gqa.W_v.weight.numel()
    kv_total = k_p + v_p
    if baseline_kv is None:
        baseline_kv = kv_total
    savings = (1 - kv_total / baseline_kv) * 100
    print(f"{name:<20} {q_p:>10,} {k_p:>10,} {v_p:>10,} {kv_total:>10,} {savings:>10.0f}%")

In [ ]:
class LlamaBlock(nn.Module):
    """Complete Llama 3 style transformer block."""
    
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int,
                 d_ff: int = None, max_len: int = 4096):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GroupedQueryAttention(d_model, n_heads, n_kv_heads, max_len)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)
    
    def forward(self, x, mask=None):
        # Pre-norm + residual
        x = x + self.attn(self.attn_norm(x), mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class LlamaModel(nn.Module):
    """Complete Llama 3 style decoder-only transformer."""
    
    def __init__(self, vocab_size: int, d_model: int = 256, n_heads: int = 8,
                 n_kv_heads: int = 4, n_layers: int = 6, d_ff: int = None,
                 max_len: int = 2048):
        super().__init__()
        self.max_len = max_len
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        # No position embedding -- RoPE handles it inside attention!
        
        self.layers = nn.ModuleList([
            LlamaBlock(d_model, n_heads, n_kv_heads, d_ff, max_len)
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying
        self.lm_head.weight = self.tok_embed.weight
        
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, input_ids, targets=None):
        B, T = input_ids.shape
        x = self.tok_embed(input_ids)
        
        causal_mask = torch.tril(torch.ones(T, T, device=input_ids.device)).unsqueeze(0).unsqueeze(0)
        
        for layer in self.layers:
            x = layer(x, causal_mask)
        
        x = self.final_norm(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
    
    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = input_ids[:, -self.max_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
        return input_ids

In [ ]:
# Instantiate and analyze the Llama-style model
VOCAB = 10000
llama = LlamaModel(
    vocab_size=VOCAB,
    d_model=256,
    n_heads=8,
    n_kv_heads=4,  # GQA: 4 KV heads for 8 query heads
    n_layers=6,
    max_len=1024,
)

total_params = sum(p.numel() for p in llama.parameters())
print(f"Llama-style model: {total_params:,} parameters")

# Breakdown
groups = {'Embedding': 0, 'Attention': 0, 'FFN': 0, 'RMSNorm': 0}
for name, p in llama.named_parameters():
    if 'tok_embed' in name:
        groups['Embedding'] += p.numel()
    elif 'attn' in name and 'norm' not in name:
        groups['Attention'] += p.numel()
    elif 'ffn' in name and 'norm' not in name:
        groups['FFN'] += p.numel()
    elif 'norm' in name:
        groups['RMSNorm'] += p.numel()

print(f"\n{'Component':<15} {'Parameters':>12} {'Fraction':>10}")
print("-" * 40)
for g, c in sorted(groups.items(), key=lambda x: -x[1]):
    print(f"{g:<15} {c:>12,} {c/total_params:>10.1%}")

# Quick forward pass
x = torch.randint(0, VOCAB, (2, 64))
logits, loss = llama(x, x)
print(f"\nForward pass OK: logits shape = {logits.shape}, loss = {loss.item():.4f}")

In [ ]:
# Quick training test on pattern data
pattern = torch.tensor(list(range(1, 51)) * 20)  # 1,2,...,50,1,2,...

def get_batch(pattern, batch_size=16, seq_len=32):
    ix = torch.randint(0, len(pattern) - seq_len - 1, (batch_size,))
    x = torch.stack([pattern[i:i+seq_len] for i in ix])
    y = torch.stack([pattern[i+1:i+seq_len+1] for i in ix])
    return x, y

small_llama = LlamaModel(
    vocab_size=100, d_model=64, n_heads=4, n_kv_heads=2,
    n_layers=2, max_len=64
)
optimizer = torch.optim.AdamW(small_llama.parameters(), lr=3e-4)

losses = []
for step in range(300):
    x, y = get_batch(pattern)
    _, loss = small_llama(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 50 == 0:
        print(f"Step {step:3d}: loss = {loss.item():.4f}")

# Generate
small_llama.eval()
prompt = torch.tensor([[1, 2, 3, 4, 5]])
gen = small_llama.generate(prompt, max_new_tokens=15, temperature=0.3)
print(f"\nPrompt:    {prompt[0].tolist()}")
print(f"Generated: {gen[0].tolist()}")
print(f"Expected:  {list(range(1, 21))}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, linewidth=1.5, alpha=0.7)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Llama-Style Model on Counting Pattern')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary

| Component | What changed | Why |
|-----------|-------------|-----|
| **RoPE** | Sinusoidal/learned -> rotation | Relative positions, no extra params, length generalization |
| **ALiBi** | Embeddings -> attention bias | Even simpler, great length generalization |
| **RMSNorm** | LayerNorm -> RMS only | Faster, fewer params, same quality |
| **SwiGLU** | GELU FFN -> gated FFN | Better quality per parameter |
| **Parallel layers** | Serial attn+FFN -> parallel | Faster on hardware, minimal quality loss |
| **GQA** | Full MHA -> grouped KV | Massive KV cache savings at inference |

### Next: `expert.ipynb`

We explore Mixture of Experts, sparse routing, and architecture scaling decisions.